# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Clasificación de relaciones con raruidol/ArgumentMining-EN-ARI-AIF-RoBERTa_L

Model page: https://huggingface.co/raruidol/ArgumentMining-EN-ARI-AIF-RoBERTa_L

# No keywords

In [1]:
import os
import re
import math
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gc

process_rel_path = r"/kaggle/working/TFM/Data/Relationships Keywords"
hf_model_repo = "raruidol/ArgumentMining-EN-ARI-AIF-RoBERTa_L"
model_name = "robertaL"  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(hf_model_repo)
model = AutoModelForSequenceClassification.from_pretrained(hf_model_repo).to(device)
model.eval()


tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/945 [00:00<?, ?B/s]

2025-08-25 01:12:30.280754: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756084350.608047      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756084350.703711      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=Tru

In [2]:
!git clone https://github.com/camipalo/TFM.git

Cloning into 'TFM'...
remote: Enumerating objects: 2370, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 2370 (delta 86), reused 98 (delta 58), pack-reused 2239 (from 2)
Receiving objects: 100% (2370/2370), 93.39 MiB | 14.96 MiB/s, done.
Resolving deltas: 100% (1976/1976), done.
Updating files: 100% (1053/1053), done.


In [3]:
# --- compute max token length across all input pairs ---
def compute_max_length(input_dir, prefix_substring, tokenizer, safety_limit=512):
    max_len = 0
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    for fn in files:
        path = os.path.join(input_dir, fn)
        df = pd.read_csv(path)
        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            continue
        for a, b in zip(df["SDGarg1"], df["SDGarg2"]):
            a = str(a) if pd.notna(a) else ""
            b = str(b) if pd.notna(b) else ""
            tokens = tokenizer(a, b, truncation=False, padding=False)["input_ids"]
            max_len = max(max_len, len(tokens))
    return min(max_len, safety_limit)

### Labels
id2label = getattr(model.config, "id2label", None) or {0: "Inference", 1: "Conflict", 2: "Rephrase"}
id2label = {int(k): str(v) for k, v in id2label.items()}

# Map model labels -> desired labels
LABEL_MAP = {
    "inference": "Support",
    "conflict": "Attack",
    "rephrase": "Rephrase",
    "none": "No Relationship",
    "0": "Support",
    "1": "Attack",
    "2": "Rephrase",
    "-1": "No Relationship",
}

VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}

def preprocess_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

@torch.no_grad()
def predict_labels(pairs):
    if not pairs:
        return []
    enc = tokenizer(
        [preprocess_text(a) for a, _ in pairs],
        [preprocess_text(b) for _, b in pairs],
        truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt"
    ).to(device)
    logits = model(**enc).logits  # [batch, num_labels]
    preds = torch.argmax(logits, dim=-1).tolist()
    raw = [id2label.get(int(p), "none") for p in preds]
    mapped = []
    for r in raw:
        key = str(r).strip().lower()
        mapped.append(LABEL_MAP.get(key, "No Relationship"))
    return mapped

def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
def classify_relationships_hf(input_dir: str, prefix_substring: str):
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    if not files:
        print(f"No CSVs found in '{input_dir}' containing '{prefix_substring}'.")
        return

    for fn in files:
        path = os.path.join(input_dir, fn)
        print(f"\nProcessing: {path}")
        df = pd.read_csv(path)

        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            print(f"  Skipped (missing SDGarg1/SDGarg2): {fn}")
            continue

        # init column
        df[rel_col] = ""

        n = len(df)
        total_done = 0
        first_examples = []

        batches = math.ceil(n / BATCH_SIZE)
        for bi in range(batches):
            s = bi * BATCH_SIZE
            e = min((bi + 1) * BATCH_SIZE, n)
            chunk = df.iloc[s:e]
            pairs = list(zip(chunk["SDGarg1"].astype(str).tolist(),
                             chunk["SDGarg2"].astype(str).tolist()))
            try:
                labels = predict_labels(pairs)
            except Exception as ex:
                print(f"  Batch {bi+1}/{batches} error: {ex}. Marking 'No Relationship'.")
                labels = ["No Relationship"] * len(pairs)

            df.loc[chunk.index, rel_col] = labels

            # collect first 5 examples
            for (a1, a2), lab in zip(pairs, labels):
                if len(first_examples) < 5:
                    first_examples.append((a1, a2, lab))
                elif len(first_examples) == 5:
                    print("Check first 5 predictions labels:\n")
                    for a1, a2, lab in first_examples:
                        print(f"- Arg1: {a1[:]} \n-Arg2: {a2[:]} \nLabel: {lab}\n\n")
                    first_examples.append((a1, a2, lab))

            total_done += len(pairs)
            if total_done % 50 < BATCH_SIZE:  
                print(f"  Progress: {total_done}/{n} relations classified...")

        # handle missing args
        mask_nan = df["SDGarg1"].isna() | df["SDGarg2"].isna()
        df.loc[mask_nan, rel_col] = "No Relationship"

        # Ensure valid set
        bad = ~df[rel_col].isin(VALID_OUT)
        if bad.any():
            df.loc[bad, rel_col] = "No Relationship"

        df.to_csv(path, index=False, encoding="utf-8")
        free_cuda()
        print(f"Saved: {path}")
        return df

## GLOBAL SDG 2023 

#### Qwen2.5 3B extraction

In [4]:
prefix = "intra_goalGLOBAL_SGD2023_qwen2.5-3b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 140

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Check first 5 predictions labels:

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
-Arg2: Investing in statistical capacity, science, and data literacy are important priorities for achieving the SDGs. 
Label: Support


- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
-Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most. 
Label: Support


- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and impleme

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
245,Many of them lack an adequately high SDG commi...,The contribution of the SDGs towards a univers...,0_11,0_26,NaN,Support,Support
139,Investing in the SDGs is an investment agenda....,The SDG Index is a flagship instrument to prom...,0_5,0_25,NaN,Support,Support
432,"All countries, poorer and richer alike, should...","build effective, accountable and inclusive ins...",16_0,16_7,NaN,Support,No Relationship
456,provide access to justice for all and build ef...,A large majority of countries – 83 percent of ...,16_5,16_6,NaN,Support,Support
346,Achieving the SDGs requires global cooperation...,The SDG Index is a flagship instrument to prom...,0_23,0_25,NaN,Support,Support


rel_robertaL
Support            380
No Relationship     85
Rephrase            25
Attack              17
Name: count, dtype: int64

In [5]:
prefix = "cross_goalGLOBAL_SGD2023_qwen2.5-3b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 153

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Check first 5 predictions labels:

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
-Arg2: Increased funding from the multilateral development banks (MDBs) and public development banks (PDBs) to low- and middle-income countries, linked to investments in the SDGs; 
Label: Support


- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
-Arg2: Many of them lack an adequately high SDG commitment, and almost all lack access to the necessary financial means to implement the SDGs. 
Label: Support


- Arg1: At their core, the SDGs are an investment agenda: it is critic

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
1731,"First, that UN Member States, at the 2023 SDG ...",The urgent need for an SDG Stimulus,0_16,17_5,NaN,Support,Support
1157,The SDG Index is a flagship instrument to prom...,"The SDG Dashboards rate rich countries, includ...",0_25,12_2,NaN,Support,Support
3615,"By design, Transformation 5 calls for regional...",The SDGs have a significant impact on public m...,11_4,16_4,NaN,Support,No Relationship
1750,The SDGs require long-term directed change and...,Reform current institutional frameworks and de...,0_18,17_4,NaN,Support,Support
2825,"Ensure access to affordable, reliable, sustain...",The SDGs also call on all countries to strengt...,7_2,10_3,NaN,Support,Support


rel_robertaL
Support            2522
No Relationship    1175
Rephrase            147
Attack              114
Name: count, dtype: int64

#### Gemma3 27B extraction

In [6]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 169

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_gemma3-27b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Despite this alarming development, the SDGs are still achievable. 
Label: Attack


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The Stimulus’ urgent object

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
1257,"At this midpoint of the 2030 Agenda, all count...",The SDG Index is also an accountability tool t...,0_44,0_48,NaN,Support,Support
1452,1. Universal quality education and innovation-...,Human capital: The skills and health of a prod...,4_0,4_4,NaN,Support,Rephrase
435,"All UN Member States should present, at regula...",The World Bank and the other MDBs should put t...,0_9,0_31,NaN,Support,Support
2219,Others include slowing or stopping the global ...,"protecting biodiversity, sustainably managing ...",13_9,13_13,NaN,Support,No Relationship
270,Further investment is needed in statistical ca...,"First, that UN Member States, at the 2023 SDG ...",0_5,0_36,NaN,Support,Support


rel_robertaL
Support            2414
No Relationship     594
Rephrase            262
Attack               84
Name: count, dtype: int64

In [7]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 179

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_gemma3-27b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Greatly increased funding for national and subnational governments and private businesses in the emerging economies, especially the low-income countries (LICs) and lower-middle-income countries (LMICs), to carry out needed SDG actions; 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: to give all people the skills and knowledge to end poverty, protect the environment, and build peaceful and inclusive societies. 
Label: No Relationship


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Although all governments are in principle committed to economic justice as enshrined in the Universal Declaration of Human Rights, and to the SDG tenets 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
3684,"At their core, the SDGs are an investment agenda.",Align private business investment flows with t...,0_19,8_1,NaN,Support,Support
33985,Even taking the net-zero pledges of many count...,SDG 17 (Partnerships for the Goals) calls on a...,13_6,17_28,NaN,Support,Support
14723,Even the most basic economic needs are current...,One of the consistent findings of the SDSN is ...,1_7,17_19,NaN,No Relationship,Support
5949,Trends on several leave-no-one-behind indicato...,Local governments have the front-line responsi...,0_39,11_3,NaN,No Relationship,No Relationship
21614,Countries must further expand and transform ed...,Promoting global cooperation and reducing geop...,4_11,17_8,NaN,Support,No Relationship


rel_robertaL
No Relationship    18791
Support            16608
Rephrase             974
Attack               459
Name: count, dtype: int64

#### Gemma3 4B extraction

In [8]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-4b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 240

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_gemma3-4b.csv
Check first 5 predictions labels:

- Arg1: the SDGs are seriously off track 
-Arg2: the SDGs are still achievable 
Label: Attack


- Arg1: the SDGs are seriously off track 
-Arg2: it is critical that UN Member States adopt and implement the SDG Stimulus 
Label: Support


- Arg1: the SDGs are seriously off track 
-Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments 
Label: Support


- Arg1: the SDGs are seriously off track 
-Arg2: Revise the credit rating system and debt sustainability metrics to facilitate long-term sustainable development 
Label: No Relationship


- Arg1: the SDGs are seriously off track 
-Arg2: Align private business investment flows with the SDGs, through improved national planning, regulation, reporting, and oversight 
Label: Support


  Progress: 250/6976 relations 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
642,All UN Member States and UN agencies can count...,It is difficult to assess whether the adoption...,0_7,0_83,NaN,Support,Support
478,Align private business investment flows with t...,In the 2030 Agenda for Sustainable Development...,0_5,0_74,NaN,Support,Rephrase
14,the SDGs are seriously off track,Preparing long-term SDG pathways to guide publ...,0_0,0_15,NaN,No Relationship,Support
1654,Investing in the SDGs,the COVID-19 pandemic has had lasting impacts ...,0_22,0_60,NaN,No Relationship,No Relationship
4231,"the transition to sustainable land use, health...",The 2021 UN Food Systems Summit raised many ur...,2_1,2_4,NaN,Support,Rephrase


rel_robertaL
Support            3734
No Relationship    2641
Rephrase            476
Attack              125
Name: count, dtype: int64

In [9]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-4b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 269

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_gemma3-4b.csv
Check first 5 predictions labels:

- Arg1: the SDGs are seriously off track 
-Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments. 
Label: Support


- Arg1: the SDGs are seriously off track 
-Arg2: Greatly increase funding to national and subnational governments and private businesses, especially in LICs and LMICs, to carry out needed SDG investments. 
Label: Support


- Arg1: the SDGs are seriously off track 
-Arg2: Revise liquidity structures for LICs and LMICs, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises; 
Label: No Relationship


- Arg1: the SDGs are seriously off track 
-Arg2: investing in statistical capacity, science, and data literacy are important priorities for achieving the SDGs 
Label: Support


- Arg1: the SDGs ar

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
35403,Expansion of private philanthropy with a focus...,partnerships to promote sustainable development,1_6,17_28,NaN,Support,No Relationship
32931,wealth creation,"Sustainable ecosystems, sustainable agricultur...",1_35,13_0,NaN,No Relationship,No Relationship
5546,Nor did countries have a common language to di...,"persistent hunger, malnutrition, and obesity",0_82,2_26,NaN,No Relationship,Support
13288,the world must devote an increased portion of ...,The poor are consequently languishing in poverty.,0_24,10_10,NaN,No Relationship,No Relationship
57796,scores are on average lowest for the second on...,Greatly increase funding to national and subna...,9_16,10_2,NaN,No Relationship,Support


rel_robertaL
No Relationship    40698
Support            27318
Rephrase            2284
Attack               934
Name: count, dtype: int64

#### Llama3.3 70B extraction

In [10]:
prefix = "intra_goalGLOBAL_SGD2023_llama3.3-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 228

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_llama3.3-70b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Since the outbreak of the pandemic in 2020 and other simultaneous crises, SDG progress has stalled globally. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The world is off track, but that is all the more reason to double down on the SDGs. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: To achieve the SDGs th

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
1138,"All UN Member States should present, at regula...",Virtually all governments of the world have em...,0_17,0_34,NaN,Support,Support
3141,SDG target 4.1 calls for universal access to 1...,"Digital technologies can raise productivity, l...",4_16,4_18,NaN,Support,No Relationship
1367,The SDGs are not only a public policy framewor...,"United Nations agencies, multilateral organiza...",0_21,0_45,NaN,Support,Support
4172,acidification of the oceans (with an increase ...,"The interconnected environmental, social, and ...",13_10,13_15,NaN,No Relationship,Support
4029,"Global warming as of 2022 stood at 1.2°C, with...",The scientific evidence points to global risks...,13_5,13_12,NaN,Support,Support


rel_robertaL
Support            4202
No Relationship    1252
Rephrase            365
Attack              132
Name: count, dtype: int64

In [11]:
prefix = "cross_goalGLOBAL_SGD2023_llama3.3-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 293

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_llama3.3-70b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Similarly, extreme poverty can lead to a collapse of tax revenues, followed by government bankruptcy and further economic collapse, a syndrome that now threatens dozens of poor countries. 
Label: No Relationship


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Most of the low-income and lower-middle income countries, home to more than the half of humanity, face major challenges in achieving most of the SDGs by 2030. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs a

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
34290,Gateways to Public Digital Learning project32 ...,Regional cooperation and sustainable developme...,4_13,17_20,NaN,Support,No Relationship
15100,Although all governments are in principle comm...,It has been announced that the current loss of...,0_26,15_4,NaN,No Relationship,No Relationship
38579,The European Union – the world’s second-larges...,Even taking the net-zero pledges of many count...,7_3,13_7,NaN,No Relationship,No Relationship
1466,National governments must ensure both domestic...,This Transformation promotes key investments i...,0_7,3_11,NaN,Support,No Relationship
58150,"First, government efforts and commitment to th...",Reform current institutional frameworks and de...,16_25,17_0,NaN,Support,No Relationship


rel_robertaL
No Relationship    30827
Support            25534
Rephrase            1332
Attack               617
Name: count, dtype: int64

#### Deepseek r1 70B extraction

In [12]:
prefix = "intra_goalGLOBAL_SGD2023_deepseek-r1-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 237

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Despite this alarming development, the SDGs are still achievable. 
Label: Attack


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The world is off track, but that is all the more reason to double down on the SDGs. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: All countries, poorer and richer alike, should use the half-way momentum to self-critically review and revise their national SDG strate

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
2606,"At the midpoint of the 2030 Agenda, all countr...","Above all, the SDGs represent an investment ag...",0_41,0_65,NaN,Support,Support
5637,National governments must ensure both domestic...,A reform of current institutional frameworks a...,17_1,17_5,NaN,Support,Support
4573,"Deep, chronic, and crippling under-investment ...","Education builds human capital, which in turn ...",10_12,10_19,NaN,Support,Support
4851,Even taking the net-zero pledges of many count...,Humanity is eroding the biological and physica...,13_5,13_8,NaN,Attack,No Relationship
6065,Failures of global governance paragraph: Secon...,"Climate change, peace, cybersecurity, reliable...",17_12,17_37,NaN,Support,Rephrase


rel_robertaL
Support            4473
No Relationship    1381
Rephrase            451
Attack              232
Name: count, dtype: int64

In [13]:
prefix = "cross_goalGLOBAL_SGD2023_deepseek-r1-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 256

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: 1. Increased funding from the multilateral develop-ment banks (MDBs) and public development banks (PDBs) to low- and middle-income countries, linked to investments in the SDGs; 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off t

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL
399,"Moreover, societal polarization, populism, and...",2. Enhancement of relief for countries facing ...,0_18,1_3,NaN,No Relationship,No Relationship
29609,Space-based technologies help address data gap...,The SDGs require long-term directed change and...,1_21,17_33,NaN,Support,No Relationship
60101,"Unless the SDGs are actively pursued, geophysi...",Commitment to multilateralism under the UN Cha...,16_6,17_34,NaN,No Relationship,No Relationship
54310,As one example: when considering consumption p...,pollution of the high seas (including plastic ...,12_9,14_2,NaN,No Relationship,Rephrase
37385,Universal quality education and innovation-bas...,pollution of the high seas (including plastic ...,4_2,14_2,NaN,No Relationship,No Relationship


rel_robertaL
No Relationship    32508
Support            25819
Rephrase            1735
Attack               929
Name: count, dtype: int64